# Nettoyage des données de transactions

Ce notebook prépare les données de ventes en vue des analyses RFM et de la segmentation client.

Le déroulement est le suivant :

1. Charger et regrouper les feuilles du fichier Excel.
2. Inspecter la structure initiale des données.
3. Supprimer les doublons et les lignes inexploitables.
4. Harmoniser les types et corriger les incohérences métier.
5. Exporter le jeu de données nettoyé.

Les cellules de code réalisent les traitements ; les textes et affichages associés expliquent leur objectif et leur résultat.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

In [2]:
sheets= pd.read_excel("../data/raw/online_retail_II.xlsx", sheet_name=None)

print(f"Feuilles excels chargées : {sheets.keys()}")

df= pd.concat(sheets.values(), ignore_index=True)

print("Données prêtes.")
display(df.head())

Feuilles excels chargées : dict_keys(['Year 2009-2010', 'Year 2010-2011'])
Données prêtes.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Chargement et regroupement

Le fichier Excel contient plusieurs feuilles. Elles sont regroupées dans un seul DataFrame afin d’obtenir une base de transactions homogène pour la suite de l’analyse.

L’aperçu permet de vérifier rapidement que les colonnes attendues sont présentes et que les données ont été correctement chargées.

## Étape 1: Inspection

In [3]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB
None


In [4]:
print(f"Dimensions initiales : {df.shape[0]:,} lignes et {df.shape[1]} colonnes.")
print(f"Nombre total de valeurs manquantes : {int(df.isna().sum().sum()):,}.")

Dimensions initiales : 1,067,371 lignes et 8 colonnes.
Nombre total de valeurs manquantes : 247,389.


## Étape 2 : Nettoyage

In [5]:
df= df.rename(columns={"Customer ID": "CustomerID"})
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Le nom `Customer ID` est renommé en `CustomerID` pour disposer d’un nom de colonne plus simple à manipuler dans les traitements Python. Aucun enregistrement n’est supprimé à cette étape.

### 1. Doublons exacts

In [6]:
cols= df.columns
nb_doublons= df.duplicated(subset=cols).sum()
print(f"Nombre de doublons exacts : {nb_doublons} ({(nb_doublons * 100 / len(df)):.2f}) %")

Nombre de doublons exacts : 34335 (3.22) %


In [7]:
print("Aperçu de quelques doublons")
display(df[df.duplicated(subset=cols)].head(20))

Aperçu de quelques doublons


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
390,489517,84951A,S/4 PISTACHIO LOVEBIRD COASTERS,1,2009-12-01 11:34:00,2.55,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
657,489529,22028,PENNY FARTHING BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984.0,United Kingdom
658,489529,22036,DINOSAUR BIRTHDAY CARD,12,2009-12-01 11:51:00,0.42,17984.0,United Kingdom


In [8]:
df.drop_duplicates(subset=cols, inplace=True)
print(F"Taille des donnéesaprès suppression des doublons : {df.shape[0]} lignes")

Taille des donnéesaprès suppression des doublons : 1033036 lignes


Les doublons exacts sont retirés car ils répètent la même transaction sur l’ensemble des colonnes. La taille affichée après suppression permet de mesurer l’effet concret de cette opération.

### 2. Valeurs manquantes

In [9]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
CustomerID     235151
Country             0
dtype: int64

In [10]:
missing = df.isna().sum()
print("Colonnes contenant des valeurs manquantes :")
print(missing[missing > 0].to_string())
print(f"Total de lignes contenant au moins une valeur manquante : {int(df.isna().any(axis=1).sum()):,}.")

Colonnes contenant des valeurs manquantes :
Description      4275
CustomerID     235151
Total de lignes contenant au moins une valeur manquante : 235,151.


In [11]:
df[df['CustomerID'].isna()].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


In [12]:
df_copy = df.copy()

df_copy.dropna(axis=0, subset=['CustomerID'], inplace=True)
df_cleaned= df_copy.reset_index(drop=True)
df_cleaned.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Les identifiants client manquants ne permettent pas de rattacher une transaction à un client. Ces lignes sont donc exclues de la base destinée à l’analyse RFM. Le nouvel index est réinitialisé pour garder une numérotation continue.

In [13]:
print(df_cleaned.isna().sum())

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
CustomerID     0
Country        0
dtype: int64


In [14]:
print(f"Taille après suppression des valeurs manquantes : {len(df_cleaned)} lignes")

Taille après suppression des valeurs manquantes : 797885 lignes


In [15]:
lignes_supprimees = len(df) - len(df_cleaned)
print(f"Lignes conservées : {len(df_cleaned):,}.")
print(f"Lignes supprimées pour CustomerID manquant : {lignes_supprimees:,}.")
print(f"Taux de conservation : {len(df_cleaned) / len(df) * 100:.2f} %.")

Lignes conservées : 797,885.
Lignes supprimées pour CustomerID manquant : 235,151.
Taux de conservation : 77.24 %.


### 3. Type de données et incohérences métier

In [16]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 797885 entries, 0 to 797884
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      797885 non-null  object        
 1   StockCode    797885 non-null  object        
 2   Description  797885 non-null  object        
 3   Quantity     797885 non-null  int64         
 4   InvoiceDate  797885 non-null  datetime64[us]
 5   Price        797885 non-null  float64       
 6   CustomerID   797885 non-null  float64       
 7   Country      797885 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 58.9+ MB


In [17]:
df_cleaned['CustomerID']= df_cleaned['CustomerID'].astype(int)
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 797885 entries, 0 to 797884
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      797885 non-null  object        
 1   StockCode    797885 non-null  object        
 2   Description  797885 non-null  object        
 3   Quantity     797885 non-null  int64         
 4   InvoiceDate  797885 non-null  datetime64[us]
 5   Price        797885 non-null  float64       
 6   CustomerID   797885 non-null  int64         
 7   Country      797885 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(2), object(3), str(1)
memory usage: 58.9+ MB


In [18]:
def get_stock_code(code : object) -> int:
    if type(code)== str:
        for char in code:
            if not str.isnumeric(char):
                code = code.replace(char, "")

        if len(code)==0:
            return np.nan

        code= str.strip(code)
        return int(code)
    if type(code)== int:
        return code

print(get_stock_code("79323W"))

79323


### Harmonisation des références produit

Les références produit peuvent contenir des lettres ou d’autres caractères. La fonction conserve uniquement la partie numérique afin d’obtenir un identifiant homogène. Une référence entièrement dépourvue de chiffres devient manquante et sera retirée à l’étape suivante.

In [19]:
df_cleaned["StockCode"]= df_cleaned["StockCode"].apply(get_stock_code)
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 797885 entries, 0 to 797884
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      797885 non-null  object        
 1   StockCode    794498 non-null  float64       
 2   Description  797885 non-null  object        
 3   Quantity     797885 non-null  int64         
 4   InvoiceDate  797885 non-null  datetime64[us]
 5   Price        797885 non-null  float64       
 6   CustomerID   797885 non-null  int64         
 7   Country      797885 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(2), object(2), str(1)
memory usage: 58.9+ MB


In [20]:
df_cleaned.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048.0,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323.0,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323.0,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041.0,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232.0,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [21]:
df_cleaned.isna().sum()

Invoice           0
StockCode      3387
Description       0
Quantity          0
InvoiceDate       0
Price             0
CustomerID        0
Country           0
dtype: int64

In [22]:
df_cleaned.dropna(subset=["StockCode"], inplace=True)
df_cleaned.shape

(794498, 8)

In [23]:
print(f"Références produit manquantes après conversion : {int(df_cleaned['StockCode'].isna().sum()):,}.")
print(f"Dimensions après suppression des références invalides : {df_cleaned.shape[0]:,} lignes et {df_cleaned.shape[1]} colonnes.")

Références produit manquantes après conversion : 0.
Dimensions après suppression des références invalides : 794,498 lignes et 8 colonnes.


In [24]:
df_cleaned["StockCode"]= df_cleaned["StockCode"].astype(int)

df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 794498 entries, 0 to 797883
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      794498 non-null  object        
 1   StockCode    794498 non-null  int64         
 2   Description  794498 non-null  object        
 3   Quantity     794498 non-null  int64         
 4   InvoiceDate  794498 non-null  datetime64[us]
 5   Price        794498 non-null  float64       
 6   CustomerID   794498 non-null  int64         
 7   Country      794498 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(3), object(2), str(1)
memory usage: 64.7+ MB


In [25]:
len(df_cleaned[df_cleaned['Price']==0.0])

62

In [26]:
df_cleaned[df_cleaned['Price']==0.0].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
3691,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126,United Kingdom
4814,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658,United Kingdom
14355,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108,United Kingdom
14356,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108,United Kingdom
24178,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070,United Kingdom
29115,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071,United Kingdom
33103,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258,United Kingdom
34296,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417,Belgium
39093,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858,United Kingdom
62932,497819,1,This is a test product.,5,2010-02-12 14:58:00,0.0,14103,United Kingdom


In [27]:
for index in list(df_cleaned[df_cleaned['Price']==0.0].index) :
    code, cid = df_cleaned.loc[index, "StockCode"], df_cleaned.loc[index, "CustomerID"]
    price= df_cleaned[(df_cleaned["StockCode"]==code) & (df_cleaned.loc[index, "CustomerID"]==cid)]["Price"].mean()
    df_cleaned.loc[index, "Price"]=price


len(df_cleaned[df_cleaned['Price']==0.0])

0

### Prix nuls et négatifs

Un prix nul ne peut pas être utilisé pour mesurer le chiffre d’affaires. Pour ces lignes, le traitement recherche le prix moyen observé pour le même produit et le même client, puis l’utilise comme remplacement. Le contrôle suivant vérifie qu’il ne reste plus de prix nul.

In [28]:
len(df_cleaned[df_cleaned["Price"]<0.0])

0

In [29]:
print(f"Prix nuls restants : {int((df_cleaned['Price'] == 0.0).sum()):,}.")
print(f"Prix négatifs restants : {int((df_cleaned['Price'] < 0.0).sum()):,}.")

Prix nuls restants : 0.
Prix négatifs restants : 0.


In [30]:
len(df_cleaned[df_cleaned["Quantity"]<0])

17596

In [31]:
df_cleaned[df_cleaned["Quantity"]<=0].head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia
179,C489449,85206,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321,Australia
185,C489449,84970,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321,Australia
196,C489459,90200,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592,United Kingdom


### Quantités et factures annulées

Une quantité négative correspond généralement à un retour ou à une annulation. Les factures dont le numéro contient `C` sont identifiées séparément afin de mesurer les annulations et de vérifier si elles sont cohérentes avec les quantités observées.

Les aperçus et les comptages ci-dessous servent à documenter ces cas particuliers avant la préparation finale.

In [32]:
print(f"Transactions avec une quantité négative : {int((df_cleaned['Quantity'] < 0).sum()):,}.")
print(f"Factures identifiées comme annulées : {int(df_cleaned['Invoice'].str.contains('C', na=False).sum()):,}.")
print(f"Factures annulées avec une quantité positive : {int(((df_cleaned['Invoice'].str.contains('C', na=False)) & (df_cleaned['Quantity'] > 0)).sum()):,}.")

Transactions avec une quantité négative : 17,596.
Factures identifiées comme annulées : 17,596.
Factures annulées avec une quantité positive : 0.


In [33]:
df_cleaned['Invoice']= df_cleaned['Invoice'].astype(str)

### Finalisation et export

Le numéro de facture est converti en texte pour conserver un format cohérent, notamment lorsque certaines factures contiennent une lettre. Une dernière inspection confirme la structure de la base avant son export au format CSV.

In [34]:
df_cleaned.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [35]:
df_cleaned.info()

<class 'pandas.DataFrame'>
Index: 794498 entries, 0 to 797883
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      794498 non-null  str           
 1   StockCode    794498 non-null  int64         
 2   Description  794498 non-null  object        
 3   Quantity     794498 non-null  int64         
 4   InvoiceDate  794498 non-null  datetime64[us]
 5   Price        794498 non-null  float64       
 6   CustomerID   794498 non-null  int64         
 7   Country      794498 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(3), object(1), str(2)
memory usage: 85.4+ MB


In [36]:
df_cleaned.to_csv("../data/processed/clean_data.csv", index=False)

Le fichier `clean_data.csv` contient maintenant la base nettoyée et prête à être réutilisée dans les étapes suivantes du projet, notamment le calcul des indicateurs RFM et la segmentation des clients.